In [ ]:
 # ============================================================
#   IRA TWEETS — UNZIP + COMBINE + LABEL
#   Run this script in the same folder as your files, OR
#   paste into Google Colab cell by cell (see CELL markers)
# ============================================================


# ────────────────────────────────────────────────────────────
# CELL 1 — Imports
# ────────────────────────────────────────────────────────────
import pandas as pd
import os
import zipfile
import glob

print("✅ Libraries imported.")


# ────────────────────────────────────────────────────────────
# CELL 2 — (Colab only) Upload your files
# ────────────────────────────────────────────────────────────
# Uncomment this block if you are running in Google Colab

# from google.colab import files
# print("Upload all IRAhandle_tweets_*.zip and IRAhandle_tweets_9.csv")
# uploaded = files.upload()
# print(f"✅ {len(uploaded)} files uploaded.")


# ────────────────────────────────────────────────────────────
# CELL 3 — Unzip all .zip files
# ────────────────────────────────────────────────────────────

# Folder where your files live (change this if needed)
# For Colab use: INPUT_DIR = "."
INPUT_DIR  = "."      # folder containing your zip + csv files
EXTRACT_DIR = "ira_data"  # folder to extract into

os.makedirs(EXTRACT_DIR, exist_ok=True)

# Find all zip files
zip_files = glob.glob(os.path.join(INPUT_DIR, "IRAhandle_tweets_*.zip"))

print(f"Found {len(zip_files)} zip file(s):")
for z in sorted(zip_files):
    print(f"  {os.path.basename(z)}")
    with zipfile.ZipFile(z, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"    ✅ Extracted")

# Also copy the standalone CSV (file 9) if it exists
standalone_csv = os.path.join(INPUT_DIR, "IRAhandle_tweets_9.csv")
if os.path.exists(standalone_csv):
    import shutil
    shutil.copy(standalone_csv, EXTRACT_DIR)
    print(f"  IRAhandle_tweets_9.csv → copied to {EXTRACT_DIR}")

print(f"\nAll files in '{EXTRACT_DIR}':")
for f in sorted(os.listdir(EXTRACT_DIR)):
    size_mb = os.path.getsize(os.path.join(EXTRACT_DIR, f)) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")


# ────────────────────────────────────────────────────────────
# CELL 4 — Read & combine all CSVs
# ────────────────────────────────────────────────────────────

csv_files = sorted(glob.glob(os.path.join(EXTRACT_DIR, "IRAhandle_tweets_*.csv")))
print(f"\nReading {len(csv_files)} CSV file(s)...")

dfs = []
for f in csv_files:
    df = pd.read_csv(f, encoding="utf-8", encoding_errors="replace", low_memory=False)
    df["source_file"] = os.path.basename(f)   # track which file each row came from
    dfs.append(df)
    print(f"  ✅ {os.path.basename(f)}: {len(df):,} rows")

combined = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows after concat: {len(combined):,}")
print(f"Columns: {list(combined.columns)}")


# ────────────────────────────────────────────────────────────
# CELL 5 — Add labels
# ────────────────────────────────────────────────────────────

# --- Label 1: political_label (detailed, human-readable) ---
category_map = {
    "LeftTroll":    "Left_Propaganda",
    "RightTroll":   "Right_Propaganda",
    "NonEnglish":   "Non_English_Propaganda",
    "Fearmonger":   "Fearmonger_Propaganda",
    "HashtagGamer": "Engagement",
    "NewsFeed":     "NewsFeed",
    "Commercial":   "Commercial",
    "Unknown":      "Unknown",
}
combined["political_label"] = (
    combined["account_category"]
    .map(category_map)
    .fillna(combined["account_category"])   # keep any unexpected value as-is
)

# --- Label 2: political_side (simplified: Left / Right / etc.) ---
def get_political_side(cat):
    cat = str(cat)
    if cat == "LeftTroll":   return "Left"
    if cat == "RightTroll":  return "Right"
    if cat == "Fearmonger":  return "Fearmonger"
    if cat == "NonEnglish":  return "NonEnglish"
    return "Other"

combined["political_side"] = combined["account_category"].apply(get_political_side)

# --- Label 3: is_troll (binary — 1 = confirmed IRA troll, 0 = other) ---
troll_categories = {"LeftTroll", "RightTroll", "Fearmonger"}
combined["is_troll"] = combined["account_category"].apply(
    lambda x: 1 if str(x) in troll_categories else 0
)

# --- Label 4: is_retweet (binary) ---
combined["is_retweet"] = (combined["retweet"] == 1).astype(int)

# --- Label 5: language_group (English vs Non-English) ---
combined["language_group"] = combined["language"].apply(
    lambda x: "English" if str(x).strip().lower() == "english" else "Non-English"
)

print("✅ Labels added:")
print(f"   political_label : {combined['political_label'].nunique()} unique values")
print(f"   political_side  : {combined['political_side'].nunique()} unique values")
print(f"   is_troll        : {combined['is_troll'].sum():,} troll rows")
print(f"   is_retweet      : {combined['is_retweet'].sum():,} retweet rows")
print(f"   language_group  : {combined['language_group'].value_counts().to_dict()}")


# ────────────────────────────────────────────────────────────
# CELL 6 — Reorder columns & deduplicate
# ────────────────────────────────────────────────────────────

final_columns = [
    # Author info
    "external_author_id", "author",
    # Tweet content
    "content",
    # Location & language
    "region", "language", "language_group",
    # Dates
    "publish_date", "harvested_date",
    # Account stats
    "following", "followers", "updates",
    # Tweet type
    "post_type", "is_retweet",
    # Original category fields (kept for reference)
    "account_type", "account_category",
    # ── NEW LABEL COLUMNS ──────────────────────────
    "political_label",   # detailed: Left_Propaganda, Right_Propaganda, etc.
    "political_side",    # simplified: Left / Right / Fearmonger / NonEnglish / Other
    "is_troll",          # binary: 1 = confirmed IRA troll
    # Misc
    "new_june_2018", "source_file",
]

combined = combined[final_columns]

# Remove duplicate tweets (same author + content + date)
rows_before = len(combined)
combined = combined.drop_duplicates(
    subset=["external_author_id", "content", "publish_date"]
).reset_index(drop=True)
rows_after = len(combined)

print(f"\n✅ Deduplication: {rows_before:,} → {rows_after:,} rows  ({rows_before - rows_after:,} duplicates removed)")


# ────────────────────────────────────────────────────────────
# CELL 7 — Summary statistics
# ────────────────────────────────────────────────────────────

print("\n" + "="*55)
print("         FINAL DATASET SUMMARY")
print("="*55)
print(f"Total rows  : {len(combined):,}")
print(f"Columns     : {len(combined.columns)}")
print()
print("── political_label distribution ──")
print(combined["political_label"].value_counts().to_string())
print()
print("── political_side distribution ──")
print(combined["political_side"].value_counts().to_string())
print()
print("── language_group distribution ──")
print(combined["language_group"].value_counts().to_string())
print()
print("── post_type distribution ──")
print(combined["post_type"].value_counts().to_string())
print("="*55)


# ────────────────────────────────────────────────────────────
# CELL 8 — Save combined CSV
# ────────────────────────────────────────────────────────────

OUTPUT_FILE = "IRA_tweets_combined_labeled.csv"

combined.to_csv(OUTPUT_FILE, index=False)
size_mb = os.path.getsize(OUTPUT_FILE) / 1e6
print(f"\n✅ Saved: '{OUTPUT_FILE}'  ({size_mb:.1f} MB)")

# --- (Colab only) Download the file ---
# Uncomment if running in Google Colab:
# from google.colab import files
# files.download(OUTPUT_FILE)
# print("📥 Download started!")

✅ Libraries imported.
Found 7 zip file(s):
  IRAhandle_tweets_1.zip
    ✅ Extracted
  IRAhandle_tweets_2.zip
    ✅ Extracted
  IRAhandle_tweets_3.zip
    ✅ Extracted
  IRAhandle_tweets_4.zip
    ✅ Extracted
  IRAhandle_tweets_5.zip
    ✅ Extracted
  IRAhandle_tweets_7.zip
    ✅ Extracted
  IRAhandle_tweets_8.zip
    ✅ Extracted
  IRAhandle_tweets_9.csv → copied to ira_data

All files in 'ira_data':
  IRAhandle_tweets_1.csv  (87.9 MB)
  IRAhandle_tweets_2.csv  (89.4 MB)
  IRAhandle_tweets_3.csv  (87.7 MB)
  IRAhandle_tweets_4.csv  (87.8 MB)
  IRAhandle_tweets_5.csv  (79.4 MB)
  IRAhandle_tweets_7.csv  (82.4 MB)
  IRAhandle_tweets_8.csv  (85.9 MB)
  IRAhandle_tweets_9.csv  (9.4 MB)

Reading 8 CSV file(s)...
  ✅ IRAhandle_tweets_1.csv: 381,016 rows
  ✅ IRAhandle_tweets_2.csv: 397,232 rows
  ✅ IRAhandle_tweets_3.csv: 368,885 rows
  ✅ IRAhandle_tweets_4.csv: 388,452 rows
  ✅ IRAhandle_tweets_5.csv: 331,821 rows
  ✅ IRAhandle_tweets_7.csv: 365,603 rows
  ✅ IRAhandle_tweets_8.csv: 378,295 row

In [ ]:
import zipfile
import os

OUTPUT_FILE = "IRA_tweets_combined_labeled.csv"
ZIP_FILE    = "IRA_tweets_combined_labeled.zip"

with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUTPUT_FILE)

print(f"✅ Zipped: '{ZIP_FILE}'  ({os.path.getsize(ZIP_FILE)/1e6:.1f} MB)")

# Uncomment if running in Google Colab:
# from google.colab import files
# files.download(ZIP_FILE)
# print("📥 Download started!")

✅ Zipped: 'IRA_tweets_combined_labeled.zip'  (166.2 MB)
